# 🐾 Animal Classification - Exploratory Data Analysis
**Author:** Vijay Bedage  
**Project:** Animal Image Classification using Transfer Learning (MobileNetV2)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

DATASET_PATH = 'Animal Classification/dataset'
CLASSES = sorted(os.listdir(DATASET_PATH))
print(f'Total Classes: {len(CLASSES)}')
print(f'Classes: {CLASSES}')

## 📊 Dataset Distribution

In [ ]:
class_counts = {}
for cls in CLASSES:
    cls_path = os.path.join(DATASET_PATH, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[cls] = len(images)

total = sum(class_counts.values())
print(f'Total Images: {total}')

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(class_counts.keys(), class_counts.values(),
              color=plt.cm.Set3(np.linspace(0, 1, len(CLASSES))), edgecolor='white')
ax.set_title(f'Image Distribution per Class (Total: {total})', fontsize=16, fontweight='bold')
ax.set_xlabel('Animal Class')
ax.set_ylabel('Number of Images')
ax.set_ylim(0, max(class_counts.values()) + 20)
for bar, val in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('results/dataset_distribution.png', dpi=150)
plt.show()

## 🖼️ Sample Images per Class

In [ ]:
ANIMAL_EMOJIS = {
    'Bear': '🐻', 'Bird': '🐦', 'Cat': '🐱', 'Cow': '🐄',
    'Deer': '🦌', 'Dog': '🐶', 'Dolphin': '🐬', 'Elephant': '🐘',
    'Giraffe': '🦒', 'Horse': '🐴', 'Kangaroo': '🦘', 'Lion': '🦁',
    'Panda': '🐼', 'Tiger': '🐯', 'Zebra': '🦓'
}

fig, axes = plt.subplots(3, 5, figsize=(18, 12))
axes = axes.flatten()

for i, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATASET_PATH, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample_img_path = os.path.join(cls_path, images[0])
    img = mpimg.imread(sample_img_path)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'{ANIMAL_EMOJIS[cls]} {cls}\n({class_counts[cls]} images)',
                      fontsize=11, fontweight='bold')

plt.suptitle('Sample Images from Each Class', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏗️ Model Architecture
We use **MobileNetV2** pretrained on ImageNet as our base model with custom classification head:

```
MobileNetV2 (frozen) → GlobalAveragePooling2D → BatchNorm → Dense(256, relu) → Dropout(0.5) → Dense(128, relu) → Dropout(0.3) → Dense(15, softmax)
```

### Why MobileNetV2?
- ✅ Lightweight — fast inference
- ✅ Pretrained on 1.2M ImageNet images
- ✅ Good accuracy with small datasets
- ✅ Mobile-friendly deployment

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model

base = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
x = base.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(15, activation='softmax')(x)
model = Model(inputs=base.input, outputs=out)

trainable = sum(1 for l in model.layers if l.trainable)
total = len(model.layers)
print(f'Total layers: {total}')
print(f'Trainable layers: {trainable}')
print(f'Frozen layers: {total - trainable}')
print(f'Total parameters: {model.count_params():,}')

## 📈 Training Results
After training, visualize results:

In [ ]:
import json

# Only works after training
if os.path.exists('results/history.json'):
    with open('results/history.json') as f:
        history = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['accuracy'], label='Train', color='royalblue')
    axes[0].plot(history['val_accuracy'], label='Validation', color='coral')
    axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history['loss'], label='Train', color='royalblue')
    axes[1].plot(history['val_loss'], label='Validation', color='coral')
    axes[1].set_title('Loss', fontsize=14, fontweight='bold')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    
    print(f"Best Val Accuracy: {max(history['val_accuracy'])*100:.2f}%")
else:
    print('Train the model first: python src/train.py')